<a href="https://colab.research.google.com/github/KunakaDK/brics_astro/blob/main/KunakaDaniel_Capstone_Project_%E2%80%93_BRICS_Astronomy_IDIA_Data_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Photometric Redshift Estimation of Galaxies Using SDSS Machine Learning Regressors

## Project Overview
This notebook performs end-to-end photometric redshift ($z$) estimation using **real astronomical data** from the Sloan Digital Sky Survey (SDSS). We implement a complete data science pipeline including:
1.  **Data Ingestion**: Programmatic SQL retrieval of galaxy magnitudes ($u, g, r, i, z$) via `astroquery`.
2.  **Feature Engineering**: Calculation of astronomical color indices.
3.  **Preprocessing**: Feature scaling and train-test splitting.
4.  **Model Optimization**: Hyperparameter tuning for Random Forest using 5-fold cross-validation.
5.  **Evaluation**: Comparison of Ridge, SVR, and Random Forest models using MAE, RMSE, and $R^2$.

## Problem Definition & Objectives
**Research Question**: Can machine learning regressors accurately estimate the spectroscopic redshift of galaxies using only broadband photometric magnitudes ($u, g, r, i, z$) from real sky surveys?

**Objectives**:
- Retrieve a real galaxy catalog from the SDSS DR16 database.
- Engineer color-index features that physically correlate with redshift due to the expansion of the universe.
- Compare the performance of linear (Ridge), kernel-based (SVR), and ensemble (Random Forest) algorithms.
- Quantify prediction error to determine suitability for large-scale cosmological surveys.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# Set plotting parameters for scientific quality
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_context("notebook", font_scale=1.1)
np.random.seed(42)

## 1. Data Acquisition & Ingestion
We utilize the `astroquery` library to perform an asynchronous SQL query on the SDSS servers. We target galaxies with clean photometry and available spectroscopic redshifts to serve as our 'ground truth' for training.

### Dataset Description & Justification
**Dataset**: SDSS DR16 Photometric Galaxy Catalog.
**Justification**: This is an authentic astronomical dataset. Photometric redshifts are essential because spectroscopic observations are time-consuming and expensive. Using 2,000 real records allows us to demonstrate the robustness of machine learning in the presence of real measurement noise and cosmic variance.

**Features**:
- `u, g, r, i, z`: Apparent magnitudes in five optical filters (Ultraviolet to Near-Infrared).
- `u_g, g_r, r_i, i_z`: Derived astronomical colors.
- `redshift`: The ground-truth spectroscopic target ($z$).

In [ ]:
# Fetching real data from SDSS DR16
from astroquery.sdss import SDSS

query = """
SELECT TOP 2000
    p.u, p.g, p.r, p.i, p.z, s.z as redshift
FROM PhotoObj AS p
   JOIN SpecObj AS s ON s.bestobjid = p.objid
WHERE
    p.type = 3 -- Galaxies
    AND s.class = 'GALAXY'
    AND s.z > 0
    AND p.u BETWEEN 0 AND 30
    AND p.g BETWEEN 0 AND 30
    AND p.r BETWEEN 0 AND 30
    AND p.i BETWEEN 0 AND 30
    AND p.z BETWEEN 0 AND 30
"""

print("Fetching real data from SDSS...")
res = SDSS.query_sql(query)
df = res.to_pandas()
df.columns = ['u', 'g', 'r', 'i', 'z', 'redshift']

display(df.head())
print(f"\nDataset shape: {df.shape}")

In [ ]:
df

## 2. Exploratory Data Analysis & Feature Engineering
Color indices represent the difference in magnitude between successive filters. They are critical because they capture the spectral energy distribution (SED) shape, which shifts with redshift.

$$\text{Color} = m_{filter1} - m_{filter2}$$

In [ ]:
# Feature Engineering: Calculating astronomical colors
df['u_g'] = df['u'] - df['g']
df['g_r'] = df['g'] - df['r']
df['r_i'] = df['r'] - df['i']
df['i_z'] = df['i'] - df['z']

print("Derived color features added.")
display(df[['u', 'u_g', 'redshift']].head())

## 3. Preprocessing & Data Splitting
We normalize features using `StandardScaler` to ensure that distance-based estimators (like SVR) or regularized models (Ridge) are not biased by the absolute magnitude scales.

In [ ]:
X = df[['u', 'g', 'r', 'i', 'z', 'u_g', 'g_r', 'r_i', 'i_z']]
y = df['redshift']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

## 4. Model Training & Hyperparameter Tuning
We optimize a Random Forest Regressor using `GridSearchCV` to find the best balance between complexity and generalization.

In [ ]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20],
    'min_samples_split': [2, 5]
}

rf_base = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(estimator=rf_base, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

best_rf = grid_search.best_estimator_
ridge = Ridge().fit(X_train_scaled, y_train)
svr = SVR(C=1.0, epsilon=0.01).fit(X_train_scaled, y_train)

print(f"Best Parameters Found: {grid_search.best_params_}")

## 5. Model Evaluation & Metrics
We assess performance using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and the R-squared ($R^2$) coefficient.

In [ ]:
models = {'Ridge': ridge, 'SVR': svr, 'Tuned Random Forest': best_rf}
results = []

for name, model in models.items():
    preds = model.predict(X_test_scaled)
    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, preds),
        'RMSE': np.sqrt(mean_squared_error(y_test, preds)),
        'R2 Score': r2_score(y_test, preds)
    })

performance_summary = pd.DataFrame(results)
display(performance_summary)

In [ ]:
def compute_astronomy_metrics(y_true, y_pred):
    """
    Calculates specialized observational cosmology metrics.
    """
    dz = y_pred - y_true
    dz_norm = dz / (1 + y_true)

    # sigma_NMAD: Normalized Median Absolute Deviation
    nmad = 1.4826 * np.median(np.abs(dz_norm - np.median(dz_norm)))

    # eta_0.15: Catastrophic Outlier Fraction
    outliers = np.abs(dz_norm) > 0.15
    eta = np.mean(outliers) * 100

    return nmad, eta

print("Metric functions defined.")

In [ ]:
# Applying specialized metrics to our model results
astro_results = []
for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    nmad, eta = compute_astronomy_metrics(y_test, y_pred)
    astro_results.append({
        'Model': name,
        'sigma_NMAD': nmad,
        'eta_0.15 (%)': eta
    })

display(pd.DataFrame(astro_results).round(4))

### 3. Astrophysical Interpretation of Feature Importance

The primary driver of photometric redshift estimation is the **Balmer break at 4000 Å**. This spectral feature is a sharp drop in flux caused by the accumulation of absorption lines from ionized metals in stellar atmospheres.

**Mechanism of Redshift Mapping:**
- At $z=0$, the 4000 Å break resides in the **$u$** and **$g$** filter interface.
- As $z$ increases, the break is redshifted progressively into the **$r, i$**, and eventually **$z$** bands.
- The Random Forest identifies which specific 'color' (the flux ratio between filters) is changing most rapidly. For example, a high $g-r$ color combined with a low $r-i$ color is a strong physical proxy for galaxies at $z \approx 0.3-0.5$, where the break has moved out of $g$ and occupies $r$.

In [ ]:
# Visualizing feature importance to check physical consistency
importances = best_rf.feature_importances_
indices = np.argsort(importances)
feature_names = X.columns

plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices], y=[feature_names[i] for i in indices], hue=[feature_names[i] for i in indices], palette='viridis', legend=False)
plt.title('Feature Importance: Identifying the Physical Drivers of Redshift', fontsize=14)
plt.xlabel('Importance (Gini Index)')
plt.show()

### 5. Scientific Discussion & Future Extensions

#### Broadband Degeneracies
A significant challenge in this analysis is the degeneracy between galaxy types. Quiescent (red) galaxies possess a strong 4000 Å break, making their $z_{\mathrm{phot}}$ highly accurate. In contrast, star-forming (blue) galaxies have flatter spectra, which can lead to larger scatter in prediction.

#### Galactic Extinction
This model assumes observed magnitudes are intrinsic. In future work, we must apply **extinction corrections** using $E(B-V)$ values from Schlegel, Finkbeiner, & Davis (SFD) dust maps to account for scattering by Milky Way dust, which reddens galaxies and can mimic higher redshifts.

#### Moving Towards Probability Density Functions $P(z)$
While this notebook provides point estimates ($z_{\mathrm{phot}}$), modern cosmological surveys like LSST and Euclid require a full **Probability Density Function $P(z)$** for each galaxy. This accounts for measurement uncertainty and avoids biases in dark energy parameter estimations. Future iterations will utilize **Quantile Regression Forests** or **Gaussian Processes** to generate these probabilistic outputs.

## 6. Scientific Conclusion & Capstone Summary

### Model Performance Summary
Our analysis compared three regression architectures for photometric redshift estimation. The **Tuned Random Forest** and **Ridge Regression** showed comparable performance, achieving an $R^2$ score of approximately **0.88**. The Mean Absolute Error (MAE) of ~0.065 indicates that our models can predict galaxy redshifts with high precision using only 5-band photometry and derived color indices.

### Key Findings
- **Physical Interpretation**: The Feature Importance plot highlights that color indices (like $u-g$ and $g-r$) are the primary predictors. This aligns with astronomical theory, as these colors capture the '4000 Å break' in galaxy spectra as it shifts through different filters due to cosmic expansion.
- **Algorithmic Insight**: While Random Forest effectively captures non-linearities, the strong performance of Ridge Regression suggests that the relationship between these specific SDSS magnitudes and redshift is largely monotonic within this range ($z < 0.8$).

### Future Steps
1. **Deep Learning**: Implementing Convolutional Neural Networks (CNNs) on raw galaxy images (rather than processed magnitudes) could further improve accuracy by capturing morphological features.
2. **Outlier Mitigation**: Incorporating uncertainty estimation (e.g., using Gaussian Processes) would be vital for handling 'catastrophic outliers' where different galaxy types appear identical in color space.



## 7. References & Data Sources
1. **SDSS DR16**: Ahumada, R., et al. "The 16th data release of the Sloan Digital Sky Surveys." *The Astrophysical Journal Supplement Series* 249.1 (2020).
2. **Scikit-Learn Documentation**: Pedregosa, F., et al. "Scikit-learn: Machine learning in Python." *Journal of Machine Learning Research* (2011).
3. **Astronomy context**: Ivezic, Ž., et al. *Statistics, Data Mining, and Machine Learning in Astronomy*. Princeton University Press (2014).